# LOG: 

/checkpoint/maui_sft/winnieyangwn/rlm_dumps/archive/gpt5/summarization/gpt-5_summarization_513_xray_2026-02-08_23-35-35_c4318c42.jsonl


In [23]:
import sys
import importlib


sys.path.append('/home/winnieyangwn/rlm/analysis')
import rlm_log_utils
importlib.reload(rlm_log_utils)
from rlm_log_utils import *

## Usage Example

Load the log file and extract key information:

# Load

In [3]:
# missing .md
missing_md = [
    "uw-madison-gi-tract-image-segmentation",
    "hotel-id-2021-fgvc8",
    "spooky-author-identification",
    "whale-categorization-playground",
    "imet-2020-fgvc7",
    "nomad2018-predict-transparent-conductors",
    "tensorflow2-question-answering",
    "champs-scalar-coupling",
    "jigsaw-unintended-bias-in-toxicity-classification",
    "osic-pulmonary-fibrosis-progression",
    "tweet-sentiment-extraction",
    "kuzushiji-recognition",
]

In [26]:
import glob
import os
import json
from pathlib import Path


def save_summarization_from_log(log_path: str) -> str | None:
    """Extract final_answer from RLM log and save as .md file.
    
    Args:
        log_path: Path to the JSONL log file from Round 1
        
    Returns:
        The final_answer string (markdown or JSON), or None if not found
    """
    log_path = log_path.strip()  # Remove any leading/trailing whitespace
    log_path_obj = Path(log_path)
    if not log_path_obj.exists():
        print(f"Log file not found: {log_path}")
        return None
    
    with open(log_path) as f:
        for line in f:
            entry = json.loads(line)
            # Find the iteration with a final_answer
            if entry.get("type") == "iteration" and entry.get("final_answer"):
                final_answer = entry["final_answer"]
                # Check for error messages that indicate REPL execution failed
                if final_answer.startswith("Error:"):
                    print(f"REPL execution failed: {final_answer}")
                    return None
                
                # Save final_answer as .md file in the same directory
                md_path = log_path_obj.with_suffix(".md")
                with open(md_path, "w") as md_file:
                    md_file.write(final_answer)
                print(f"Saved final_answer to {md_path} ({len(final_answer)} chars)")
                
                return final_answer
    
    print("No final_answer found in log")
    return None


def process_missing_md_tasks(
    missing_md: list[str],
    log_dir: str = "/checkpoint/maui_sft/winnieyangwn/rlm_dumps/summarization/mle-30",
    run_id: int = 513,
    model_name: str = "gpt-5"
) -> dict:
    """
    Find log files for each task_name in missing_md, extract final_answer,
    and save summarization result if possible.
    
    Args:
        missing_md: List of task names to process
        log_dir: Directory containing log files
        run_id: Run ID for matching log files
        model_name: Model name prefix for matching log files
        
    Returns:
        Dict with 'success', 'failed', and 'skipped' lists containing task results
    """
    results = {
        "success": [],
        "failed": [],
        "skipped": []
    }
    
    for task_name in missing_md:
        print(f"\n{'='*60}")
        print(f"Processing: {task_name}")
        print(f"{'='*60}")
        
        # Find log files matching the pattern (only .jsonl files)
        log_path_prefix = f"{log_dir}/{model_name}_summarization_{run_id}_{task_name}"
        matching_logs = glob.glob(f"{log_path_prefix}*.jsonl")
        
        if not matching_logs:
            print(f"❌ No log files found matching prefix: {log_path_prefix}")
            results["failed"].append({
                "task_name": task_name,
                "reason": f"No log files found matching prefix: {log_path_prefix}"
            })
            continue
        
        # Get the most recent log file by modification time
        log_path = max(matching_logs, key=os.path.getmtime)
        print(f"Found {len(matching_logs)} matching log file(s)")
        print(f"Using most recent: {log_path}")
        
        # Load log and check for final_answer
        try:
            entries = load_rlm_log(log_path)
            iterations = entries[1:]
            final_answer = get_final_answer(iterations)
            
            if final_answer is None:
                # Skip tasks with no final answer
                print(f"⏭️ Skipping: No final answer found in log")
                results["skipped"].append({
                    "task_name": task_name,
                    "reason": "final_answer is None",
                    "log_path": log_path
                })
                continue
            
            print(f"✓ Final answer found ({len(final_answer)} chars)")
            
            # Check if it's an error
            if final_answer.startswith("Error:"):
                print(f"❌ Final answer is an error: {final_answer[:200]}...")
                results["failed"].append({
                    "task_name": task_name,
                    "reason": f"Final answer is an error: {final_answer[:200]}",
                    "log_path": log_path
                })
                continue
            
            # Save using save_summarization_from_log
            saved_answer = save_summarization_from_log(log_path)
            
            if saved_answer:
                print(f"✓ Successfully saved summarization for {task_name}")
                results["success"].append({
                    "task_name": task_name,
                    "log_path": log_path,
                    "answer_length": len(saved_answer)
                })
            else:
                print(f"❌ Failed to save summarization for {task_name}")
                results["failed"].append({
                    "task_name": task_name,
                    "reason": "save_summarization_from_log returned None",
                    "log_path": log_path
                })
                
        except Exception as e:
            print(f"❌ Error processing log: {e}")
            results["failed"].append({
                "task_name": task_name,
                "reason": f"Exception: {str(e)}",
                "log_path": log_path if 'log_path' in dir() else None
            })
    
    # Print summary
    print(f"\n{'='*60}")
    print("SUMMARY")
    print(f"{'='*60}")
    print(f"✓ Success: {len(results['success'])} tasks")
    print(f"⏭️ Skipped: {len(results['skipped'])} tasks")
    print(f"❌ Failed: {len(results['failed'])} tasks")
    
    if results["skipped"]:
        print("\nSkipped tasks (no final_answer):")
        for item in results["skipped"]:
            print(f"  - {item['task_name']}")
    
    if results["failed"]:
        print("\nFailed tasks:")
        for item in results["failed"]:
            print(f"  - {item['task_name']}: {item['reason'][:80]}...")
    
    return results


# Run the function on missing_md
results = process_missing_md_tasks(missing_md)


Processing: uw-madison-gi-tract-image-segmentation
Found 3 matching log file(s)
Using most recent: /checkpoint/maui_sft/winnieyangwn/rlm_dumps/summarization/mle-30/gpt-5_summarization_513_uw-madison-gi-tract-image-segmentation_2026-02-15_06-24-37_855c8291.jsonl
✓ Final answer found (790 chars)
Saved final_answer to /checkpoint/maui_sft/winnieyangwn/rlm_dumps/summarization/mle-30/gpt-5_summarization_513_uw-madison-gi-tract-image-segmentation_2026-02-15_06-24-37_855c8291.md (790 chars)
✓ Successfully saved summarization for uw-madison-gi-tract-image-segmentation

Processing: hotel-id-2021-fgvc8
Found 3 matching log file(s)
Using most recent: /checkpoint/maui_sft/winnieyangwn/rlm_dumps/summarization/mle-30/gpt-5_summarization_513_hotel-id-2021-fgvc8_2026-02-15_06-25-32_b8c26d73.jsonl
✓ Final answer found (238213 chars)
Saved final_answer to /checkpoint/maui_sft/winnieyangwn/rlm_dumps/summarization/mle-30/gpt-5_summarization_513_hotel-id-2021-fgvc8_2026-02-15_06-25-32_b8c26d73.md (238213 

In [25]:
import glob
import os

run_id = 513
model_name = "gpt-5"
task_name = missing_md[1]
log_dir = "/checkpoint/maui_sft/winnieyangwn/rlm_dumps/summarization/mle-30"
codebase_extensions = [".py", ".md", ".yaml"]

# Option 1: Provide a specific log name to load directly
log_name = None  # e.g., "gpt-5_summarization_513_20260208_123456.jsonl"

if log_name:
    # Directly load the specified log file
    LOG_PATH = os.path.join(log_dir, log_name)
    if not os.path.exists(LOG_PATH):
        raise FileNotFoundError(f"Specified log file not found: {LOG_PATH}")
    print(f"Loading specified log: {LOG_PATH}")
else:
    # Use underscore after run_id to avoid matching "summarization_comparison_*" logs
    LOG_PATH_PREFIX = f"{log_dir}/{model_name}_summarization_{run_id}_{task_name}"

    # Find all log files matching the prefix pattern
    matching_logs = glob.glob(f"{LOG_PATH_PREFIX}*")

    if not matching_logs:
        raise FileNotFoundError(f"No log files found matching prefix: {LOG_PATH_PREFIX}")

    # Get the most recent log file by modification time
    LOG_PATH = max(matching_logs, key=os.path.getmtime)
    print(f"Found {len(matching_logs)} matching log file(s)")
    print(f"Loading most recent: {LOG_PATH}")

# Load the log - first entry is metadata, rest are iterations
entries = load_rlm_log(LOG_PATH)
metadata = entries[0]
iterations = entries[1:]

print(f"Loaded {len(iterations)} iterations")

Found 4 matching log file(s)
Loading most recent: /checkpoint/maui_sft/winnieyangwn/rlm_dumps/summarization/mle-30/gpt-5_summarization_513_hotel-id-2021-fgvc8_2026-02-15_06-25-32_b8c26d73.md


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

# Metadata

In [13]:
# View metadata
print("=== METADATA ===")
for k, v in metadata.items():
    if k != "backend_kwargs":
        print(f"{k}: {v}")

=== METADATA ===
type: metadata
timestamp: 2026-02-15T06:25:32.824408
root_model: gpt-5
max_depth: 2
max_iterations: 10
backend: azure_openai
environment_type: local
environment_kwargs: {'setup_code': '\nimport pandas as pd\n\n# Load rollout data as DataFrame\nrollout_df = pd.read_json(\'/checkpoint/maui_sft/winnieyangwn/amaia_dumps/513/trajectories/513_metadata.jsonl\', lines=True)\nprint(f"Loaded {len(rollout_df)} total rollouts")\n\n# Filter to specific task\nrollout_df = rollout_df[rollout_df[\'task_name\'] == \'hotel-id-2021-fgvc8\']\nprint(f"Filtered to {len(rollout_df)} rollouts for task: hotel-id-2021-fgvc8")\n'}
other_backends: None


In [14]:

# Compare with timestamp-based runtime
runtime = get_total_runtime(entries)
print(f"Timestamp-based runtime: {runtime.total_seconds():.2f}s")

Timestamp-based runtime: 294.11s


In [15]:
# Check number of iterations actually taken by model
num_iterations = len(iterations)
print(f"Number of iterations taken: {num_iterations}")

# You can also use extract_all for a comprehensive summary
summary = extract_all(LOG_PATH)
print(f"Number of iterations (from extract_all): {summary['num_iterations']}")

Number of iterations taken: 3
Number of iterations (from extract_all): 3


In [16]:
# Get the final answer
final_answer = get_final_answer(iterations)
print("=== FINAL ANSWER ===")
print(final_answer if final_answer else "No final answer found")
# print(f"\n(Total length: {len(final_answer) if final_answer else 0} chars)")

=== FINAL ANSWER ===
Part 0: Task Analysis
1) Problem Type
- Fine-grained multi-class image classification/identification (hotel ID), typically approached as image retrieval/metric learning with ranked outputs (top-k).

2) Domain
- Computer vision for public safety/anti-human-trafficking: hotel room recognition to aid investigations.

3) Input Format
- Images: RGB photos of hotel room interiors.
  - Train: 97k+ images from ~7,700 hotels (train_images), organized in chain-specific subfolders.
  - Test: Hidden set (~13k images; 3 samples provided).
- Training metadata (train.csv):
  - image (filename)
  - chain (ID; 0 = not in a chain/unknown; not available for test)
  - hotel_id (target class)
  - timestamp (train only)
- Submission/output: For each test image, a space-delimited, ranked list (up to 5) of predicted hotel_id values.

4) Evaluation Metric
- Mean Average Precision at 5 (MAP@5): averages per-image average precision over all test images, considering up to the top 5 prediction

# Iteration #1

In [17]:
iteration_id = 1
iteration = iterations[iteration_id] if iterations else None

if iteration:
    print(f"=================  Iteration {iteration_id} (#{iteration.get('iteration', 'N/A')}) =====================")
    print(f"Keys in  iterxation {iteration_id} : {list(iteration.keys())}")
    

    # Check LLM response
    print(f"\n================= LLM Response in Iteration {iteration_id} =====================")
    response = iteration.get("response", "")
    print(f"Response:\n...{response}" )


    # Check code blocks in iteration
    code_blocks = iteration.get("code_blocks", [])
    print(f"\nNumber of code blocks: {len(code_blocks)}")
    
    for i, block in enumerate(code_blocks):
        print(f"\n======================================== Code Block {i} =======================================")
        # code = block.get("code", "")
        # print(f"\n----------------------- Code Block {i} Code ------------------")

        # print(f"Code: {code}")
        
        result = block.get("result", {})
        stdout = result.get("stdout", "")
        stderr = result.get("stderr", "")
        
        if stdout:
            print(f"\n----------------------- Code Block {i} Stdout ------------------")

            print(f"\nStdout:\n{stdout}" )
        if stderr:
            print(f"\n----------------------- Code Block {i} Stderr ------------------")
            print(f"\nStderr:\n{stderr}")
    

else:
    print("No iterations found!")

=================  Iteration 1 (#2) =====================
Keys in  iterxation 1 : ['type', 'iteration', 'timestamp', 'prompt', 'response', 'code_blocks', 'final_answer', 'iteration_time']

================= LLM Response in Iteration 1 =====================
Response:
...```repl
import math

# Part 0: Task Analysis from the first task_description
task_desc = rollout_df["task_description"].iloc[0]

part0_prompt = f"""You are given the Kaggle task description markdown for hotel-id-2021-fgvc8. Extract the following clearly and concisely:
1. Problem Type (Classification/Regression/Object Detection/etc.)
2. Domain (application area)
3. Input Format (Images/Tabular/Text/etc. and details)
4. Evaluation Metric (name and what it optimizes for)
5. Key Challenges (bullet list)

Task description:
{task_desc}
"""

# Part 1: Summaries for valid submissions
hotel_mask = rollout_df["task_name"] == "hotel-id-2021-fgvc8"
valid_mask = rollout_df["valid_submission"] == True
hotel_valid_df = rollout_df[hotel

In [18]:
iteration_id = 2
iteration = iterations[iteration_id] if iterations else None

if iteration:
    print(f"=================  Iteration {iteration_id} (#{iteration.get('iteration', 'N/A')}) =====================")
    print(f"Keys in  iterxation {iteration_id} : {list(iteration.keys())}")
    

    # Check LLM response
    print(f"\n================= LLM Response in Iteration {iteration_id} =====================")
    response = iteration.get("response", "")
    print(f"Response:\n...{response}" )


    # Check code blocks in iteration
    code_blocks = iteration.get("code_blocks", [])
    print(f"\nNumber of code blocks: {len(code_blocks)}")
    
    for i, block in enumerate(code_blocks):
        print(f"\n======================================== Code Block {i} =======================================")
        # code = block.get("code", "")
        # print(f"\n----------------------- Code Block {i} Code ------------------")

        # print(f"Code: {code}")
        
        result = block.get("result", {})
        stdout = result.get("stdout", "")
        stderr = result.get("stderr", "")
        
        if stdout:
            print(f"\n----------------------- Code Block {i} Stdout ------------------")

            print(f"\nStdout:\n{stdout}" )
        if stderr:
            print(f"\n----------------------- Code Block {i} Stderr ------------------")
            print(f"\nStderr:\n{stderr}")
    

else:
    print("No iterations found!")

=================  Iteration 2 (#3) =====================
Keys in  iterxation 2 : ['type', 'iteration', 'timestamp', 'prompt', 'response', 'code_blocks', 'final_answer', 'iteration_time']

================= LLM Response in Iteration 2 =====================
Response:
...FINAL_VAR(final_answer)

Number of code blocks: 0


In [19]:
iteration_id = 3
iteration = iterations[iteration_id] if iterations else None

if iteration:
    print(f"=================  Iteration {iteration_id} (#{iteration.get('iteration', 'N/A')}) =====================")
    print(f"Keys in  iterxation {iteration_id} : {list(iteration.keys())}")
    

    # Check LLM response
    print(f"\n================= LLM Response in Iteration {iteration_id} =====================")
    response = iteration.get("response", "")
    print(f"Response:\n...{response}" )


    # Check code blocks in iteration
    code_blocks = iteration.get("code_blocks", [])
    print(f"\nNumber of code blocks: {len(code_blocks)}")
    
    for i, block in enumerate(code_blocks):
        print(f"\n======================================== Code Block {i} =======================================")
        # code = block.get("code", "")
        # print(f"\n----------------------- Code Block {i} Code ------------------")

        # print(f"Code: {code}")
        
        result = block.get("result", {})
        stdout = result.get("stdout", "")
        stderr = result.get("stderr", "")
        
        if stdout:
            print(f"\n----------------------- Code Block {i} Stdout ------------------")

            print(f"\nStdout:\n{stdout}" )
        if stderr:
            print(f"\n----------------------- Code Block {i} Stderr ------------------")
            print(f"\nStderr:\n{stderr}")
    

else:
    print("No iterations found!")

IndexError: list index out of range